In [2]:
# 1. 시스템 패키지 리스트 업데이트
!apt-get update

# 2. Tesseract OCR 엔진과 한국어 언어팩 설치
!apt-get install -y tesseract-ocr tesseract-ocr-kor

# 3. OCR 및 데이터 처리에 필요한 Python 라이브러리 설치
!pip install pytesseract opencv-python-headless pandas

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [1,801 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,067 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,040 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [3,350 kB]
Hit:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:13 http://security.ubuntu.com/ubunt

In [3]:
import cv2
import numpy as np
import pytesseract
import pandas as pd
import json
from collections import OrderedDict

# 이미지 경로
img_path = 'sample_data/sample_7.png'
img = cv2.imread(img_path)

# 행/열 수 (데이터 부분만)
n_rows, n_cols = 8, 28

# 셀 크기 계산 (상단, 좌측 인덱스까지 감안해서 +1)
cell_h = img.shape[0] // (n_rows + 1)
cell_w = img.shape[1] // (n_cols + 1)

def ocr_cell(cell_img):
    gray = cv2.cvtColor(cell_img, cv2.COLOR_BGR2GRAY)
    # 글씨 잘 보이게 확대
    gray = cv2.resize(gray, None, fx=2, fy=2, interpolation=cv2.INTER_LINEAR)
    _, threshed = cv2.threshold(gray, 180, 255, cv2.THRESH_BINARY)
    # D/N/- 만 인식하게 제한
    config = "--psm 10 -c tessedit_char_whitelist=DN-"
    text = pytesseract.image_to_string(threshed, lang='eng', config=config)
    return text.strip().replace('\n', '').replace(' ', '')

result = OrderedDict()
for row in range(n_rows):
    row_dict = OrderedDict()
    for col in range(n_cols):
        # 표 라인/테두리 겹침 방지로 패딩 살짝 추가
        x1 = (col + 1) * cell_w + 2
        x2 = (col + 2) * cell_w - 2
        y1 = (row + 1) * cell_h + 2
        y2 = (row + 2) * cell_h - 2
        cell_img = img[y1:y2, x1:x2]

        # 빈칸 판별: 흰색 비율이 높으면 '-'
        cell_gray = cv2.cvtColor(cell_img, cv2.COLOR_BGR2GRAY)
        white_ratio = np.sum(cell_gray > 220) / cell_gray.size
        if white_ratio > 0.85:
            val = '-'
        else:
            val = ocr_cell(cell_img)
            if len(val) == 0 or (val not in ['D', 'N', '-']):
                val = '-'
        row_dict[str(col)] = val
    result[str(row)] = row_dict

# 결과를 판다스로 보여주거나
df = pd.DataFrame(result).T
print(df)

# JSON으로 저장/출력
json_result = json.dumps(result, ensure_ascii=False, indent=2)
print(json_result)


   0  1  2  3  4  5  6  7  8  9  ... 18 19 20 21 22 23 24 25 26 27
0  D  D  D  D  -  -  -  D  D  D  ...  D  D  D  D  -  -  -  N  N  N
1  -  -  -  D  D  D  D  N  N  -  ...  -  -  -  -  -  -  D  D  D  D
2  D  D  N  N  -  -  -  D  D  D  ...  D  D  D  N  N  -  -  D  D  D
3  N  N  -  -  D  D  D  D  D  D  ...  -  -  -  D  D  N  N  -  -  -
4  D  D  D  -  -  -  -  -  -  -  ...  N  N  N  -  -  D  D  -  -  -
5  -  -  -  -  N  N  N  -  -  -  ...  -  -  -  D  D  -  -  -  -  -
6  -  -  -  -  -  -  -  -  -  -  ...  -  -  -  -  -  -  -  -  -  -
7  -  -  -  -  -  -  -  -  -  N  ...  -  -  -  -  -  -  -  -  -  -

[8 rows x 28 columns]
{
  "0": {
    "0": "D",
    "1": "D",
    "2": "D",
    "3": "D",
    "4": "-",
    "5": "-",
    "6": "-",
    "7": "D",
    "8": "D",
    "9": "D",
    "10": "D",
    "11": "D",
    "12": "-",
    "13": "-",
    "14": "-",
    "15": "-",
    "16": "D",
    "17": "D",
    "18": "D",
    "19": "D",
    "20": "D",
    "21": "D",
    "22": "-",
    "23": "-",
    "24": "-"